# Human in the Loop

This concepts comes when we need human interventions like sending an email, payments, accessing social security numbers. All these require Human eye to approve or reject, for that we require a middleware function for that. 

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body"""
    # fake email sending
    return f"Email Sent"

In [5]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str


agent = create_agent(
    model="claude-sonnet-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution required approval"
        )
    ]
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Sean, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all! '
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                 

In [8]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem at all! Thanks for letting me know. I'm flexible with my schedule. What time works better for you tomorrow, or would you prefer to move it to another day?\n\nLet me know what works best.\n\nBest,\nSean"}, 'description': 'Tool execution required approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem at all! Thanks for letting me know. I\'m flexible with my schedule. What time works better for you tomorrow, or would you prefer to move it to another day?\\n\\nLet me know what works best.\\n\\nBest,\\nSean"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='8b429a363af8decbaae2657079e72c0e')]


In [ ]:
print()